In [ ]:
!pip -q install transformers accelerate openpyxl pandas sentencepiece

import pandas as pd
import torch
import re

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)


from google.colab import files

uploaded = files.upload()



MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)



SYSTEM = """
Sen uzman bir hukuk asistanısın.

Türkçe cevap ver.

Kısa ve net cevap ver.

Emin değilsen bunu belirt.
"""

def ask(question):

    prompt = tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": SYSTEM
            },
            {
                "role": "user",
                "content": question
            }
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    answer = pipe(prompt)[0]["generated_text"]

    answer = re.sub(
        r"<think>.*?</think>",
        "",
        answer,
        flags=re.S
    )

    return answer.strip()



df = pd.read_excel(
    "Sorular.xlsx",
    header=None
)

questions = df.iloc[:, 0]

print(f"\nToplam {len(questions)} soru bulundu.\n")


results = []

for i, question in enumerate(questions, start=1):

    question = str(question)

    print(f"Soru {i}/{len(questions)}")

    answer = ask(question)

    results.append({
        "question": question,
        "answer": answer
    })



result_df = pd.DataFrame(results)

display(result_df)

result_df.to_csv(
    "baseline_results.csv",
    index=False,
    encoding="utf-8-sig"
)





files.download("baseline_results.csv")